# Day 1 — PyTorch → HuggingFace

환경 점검 셀. 자유롭게 수정/추가하면서 진행하세요.

In [8]:
import torch
print(torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu")

2.13.0+cu130
CUDA available: True
Orin


## 0. 방금 뜬 경고, 무시해도 될까?

위 셀을 실행하면 이런 경고가 뜹니다:

> No published PyTorch CUDA builds for release 2.13.0+cu130 support this GPU (CC 8.7)

**결론부터: 이 튜토리얼 진행에는 문제 없습니다.** 아래 셀들에서 텐서 연산, LLM 로딩, 텍스트 생성까지 이 경고와 함께 전부 정상 동작하는 걸 직접 확인했습니다.

**경고의 정확한 의미**: 지금 설치된 torch는 aarch64(Jetson과 같은 ARM64)용으로 올바르게 설치되어 있습니다 — 만약 잘못된 아키텍처(x86_64)였다면 아예 import 단계에서 실행 자체가 안 됩니다. 이 경고는 그것과는 다른 얘기로, torch가 미리 컴파일해둔 GPU 커널 목록에 Orin의 정확한 Compute Capability인 8.7이 명시적으로 포함되지 않았다는 뜻입니다. 대신 가장 가까운 CC(8.0)용 PTX 코드를 런타임에 JIT 컴파일해서 사용하기 때문에, 최초 실행 시 약간 느리고 경고가 뜨지만 결과는 정상입니다.

성능이 정말 중요해지는 건 Day 2(추론 속도)·Day 4(양자화)부터입니다. 지금은 넘어가도 좋습니다.

## 1. PyTorch 복습: 텐서와 GPU

HuggingFace로 넘어가기 전에 PyTorch의 핵심만 짚고 갑니다: 텐서를 만들고, `.to("cuda")`로 GPU에 올리고, 연산하는 것.

In [9]:
import torch

a = torch.randn(3, 3)
print("CPU 텐서:\n", a)

a_gpu = a.to("cuda")
b_gpu = torch.eye(3, device="cuda")
# print("b_gpu", b_gpu)
c = a_gpu @ b_gpu  # 행렬곱
print("\nGPU 연산 결과:\n", c)
print("\ndevice:", c.device)

CPU 텐서:
 tensor([[ 0.6531, -0.4418, -0.0490],
        [ 0.2967, -0.3117,  1.5519],
        [ 0.0913,  0.7366, -0.5803]])

GPU 연산 결과:
 tensor([[ 0.6531, -0.4418, -0.0490],
        [ 0.2967, -0.3117,  1.5519],
        [ 0.0913,  0.7366, -0.5803]], device='cuda:0')

device: cuda:0


## 2. HuggingFace가 해결하는 문제

순수 PyTorch로 트랜스포머 모델을 쓰려면 직접 준비해야 할 게 많습니다:

- 모델 아키텍처 코드 (attention, layer norm, position embedding, ...)
- 사전학습된 가중치를 그 아키텍처에 정확히 맞춰 불러오기
- 토크나이저 (텍스트 ↔ 토큰 ID 변환), 모델마다 다른 vocab/규칙
- 생성 로직 (greedy/beam/sampling, EOS 처리 등)

HuggingFace `transformers`는 이걸 `AutoTokenizer` / `AutoModelForCausalLM` 두 줄로 압축합니다. 모델 이름만 바꾸면 수만 개의 사전학습 모델을 동일한 코드로 쓸 수 있습니다.

## 3. Tokenizer + Model 직접 다루기

작은 모델(`Qwen2.5-0.5B-Instruct`, 5억 파라미터)로 시작합니다. Jetson Orin Nano에서 fp16으로 로드하면 GPU 메모리 약 1GB만 사용합니다.

처음 실행하면 모델을 다운로드하느라 다소 걸립니다 (이후엔 캐시되어 빠릅니다).

In [10]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, dtype=torch.float16).to("cuda")

messages = [{"role": "user", "content": "PyTorch와 HuggingFace의 관계를 한 문장으로 설명해줘"}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

output_ids = model.generate(**inputs, max_new_tokens=64)
response = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(response)

/home/manager/mlops-lab-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1225.72it/s]


PyTorch과 Hugging Face는 서로 빠르게 통합될 수 있도록 협력하고 있습니다. PyTorch는 Hugging Face의 Natural Language Processing 모델을 개발하고 훈련하는 데 사용되는 라이브러리입니다. 이 라이브


**방금 한 일 분해:**

1. `AutoTokenizer.from_pretrained` — 이 모델 전용 토크나이저 로드
2. `AutoModelForCausalLM.from_pretrained(..., dtype=torch.float16)` — 가중치를 fp16으로 불러와 GPU로 이동 (메모리 절반으로 절약)
3. `apply_chat_template` — 모델이 학습된 대화 형식(예: `<|im_start|>user ... <|im_end|>`)으로 프롬프트 포맷팅
4. `tokenizer(...)` — 텍스트 → 토큰 ID 텐서
5. `model.generate(...)` — 토큰을 한 개씩 예측하며 이어붙임 (autoregressive)
6. `tokenizer.decode(...)` — 토큰 ID → 텍스트, 입력 프롬프트 길이만큼 잘라내고 새로 생성된 부분만 출력

> 참고: `torch_dtype` 인자는 최신 transformers에서 deprecated이고 `dtype`으로 이름이 바뀌었습니다.

## 4. `pipeline()`으로 더 간단히

위 5단계를 매번 반복하지 않도록, HuggingFace는 `pipeline()`이라는 한 줄짜리 고수준 API를 제공합니다.

In [11]:
from transformers import pipeline

pipe = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct", dtype=torch.float16, device="cuda")

messages = [{"role": "user", "content": "Jetson Orin Nano로 LLM을 공부하는 이유를 한 문장으로"}]
result = pipe(messages, max_new_tokens=48)
print(result[0]["generated_text"][-1]["content"])

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1122.70it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=48) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force clea

Jetson Orin Nano는 Jetson Nano의 개발자들에게 특별히 주어진 기술적 요구에 맞춰 설계된 고급형 모델입니다. 이 모델은 AI 기반의 로봇 �


## 5. 관찰: 속도와 메모리

직접 실행해보며 확인한 수치입니다 (참고용, 실행마다 조금씩 달라질 수 있음):

| 항목 | 값 |
|---|---|
| 모델 | Qwen2.5-0.5B-Instruct (fp16) |
| GPU 메모리 사용량 | 약 1GB |
| 첫 생성 (콜드 스타트, JIT 컴파일 포함) | ~2 tok/s |
| 이후 생성 (워밍업 후) | ~8 tok/s |

속도가 들쭉날쭉하고 첫 호출이 느린 이유가 바로 위(0번)에서 본 CC 8.7 PTX JIT 컴파일 때문입니다 — 커널이 미리 컴파일되어 있지 않아 처음 실행할 때 컴파일 비용이 듭니다.

- **Day 2**에서 이 지연(TTFT)과 토큰당 생성 속도(TPOT)를 정확히 측정하고 원인을 더 파고듭니다.
- **Day 4**에서 양자화(INT8/INT4)로 메모리와 속도를 함께 줄이는 방법을 다룹니다.

## 직접 해보기

아래 빈 셀에서 자유롭게 실험해보세요:

- 프롬프트를 바꿔서 결과가 어떻게 달라지는지 확인
- `max_new_tokens`를 늘리거나 줄여서 속도 변화 체감
- 다른 작은 모델로 교체해보기 (예: `Qwen/Qwen2.5-1.5B-Instruct`, 메모리 사용량 비교)
- `model.generate(..., temperature=0.9, do_sample=True)`처럼 샘플링 옵션 추가해보기

In [12]:
import time
import torch

# 1. 원하는 질문(프롬프트)으로 수정해보세요!
custom_prompt = "인공지능이 임베디드 장치(Jetson)에서 동작할 때의 장점을 3가지로 요약해줘."

messages = [{"role": "user", "content": custom_prompt}]
formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(formatted_prompt, return_tensors="pt").to("cuda")

# 2. 다양한 생성 옵션을 변경해가며 실험해보세요.
# - max_new_tokens: 생성할 답변의 길이 제한 (더 길게 만들고 싶다면 늘려보세요)
# - do_sample: True로 설정하면 매번 다른(창의적인) 답변이 나오고, False면 항상 고정된 답변이 나옵니다.
# - temperature: 값이 높을수록(예: 0.9) 다양하고 창의적인 답변이, 낮을수록(예: 0.2) 일관되고 정확한 답변이 나옵니다.
# - top_p: 누적 확률 필터링으로 값이 높을수록 더 다양한 단어를 선택합니다.

start_time = time.time()

output_ids = model.generate(
    **inputs,
    max_new_tokens=150,      # 답변 길이를 150 토큰으로 넉넉히 설정
    do_sample=True,          # 샘플링 활성화 (매번 다양한 답변 생성)
    temperature=0.7,         # 창의성 조절 (0.0 ~ 1.0)
    top_p=0.9,               # 상위 90% 확률 단어 풀에서 선택
    repetition_penalty=1.1   # 동일 단어 반복 방지
)

end_time = time.time()

response = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("=== [질문] ===")
print(custom_prompt)
print("\n=== [답변] ===")
print(response)
print("\n" + "=" * 20)
print(f"생성 소요 시간: {end_time - start_time:.2f}초")

=== [질문] ===
인공지능이 임베디드 장치(Jetson)에서 동작할 때의 장점을 3가지로 요약해줘.

=== [답변] ===
1. 인공지능의 핵심 기술을 효과적으로 활용하기 위해 필요한 빠른 처리 속도를 갖추기 위한 장점입니다.
2. 대형 데이터와 비대칭성에 의존하지 않아 실시간 데이터 수집과 처리를 가능하게 하는 장점입니다.
3. 인공지능의 정보 입력 및 분석 능력을 확보하기 위해 필요한 고도의 정보 입력 및 분석 기능을 제공하는 장점입니다.

생성 소요 시간: 8.98초
